In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import re

dbname = 'suumo.db'
conn = sqlite3.connect(dbname)
cur = conn.cursor()

cur.execute('DROP TABLE IF EXISTS properties')
cur.execute('''
    CREATE TABLE properties (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        price INTEGER,
        age INTEGER,
        floor_plan TEXT
    )
''')
conn.commit()

base_url = "https://suumo.jp/jj/chintai/ichiran/FR301FC001/?ar=040&bs=040&ta=15&sc=15202&cb=0.0&ct=9999999&et=9999999&cn=9999999&mb=0&mt=9999999&shkr1=03&shkr2=03&shkr3=03&shkr4=03&fw2="

def get_data():
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    for page in range(1, 12):
        print(f"{page}回目 ")
        url = f"{base_url}&page={page}"
        
        try:
            res = requests.get(url, headers=headers)
            res.encoding = 'utf-8'
            
            if res.status_code != 200:
                print(f"Error: {res.status_code}")
                continue

            soup = BeautifulSoup(res.text, 'html.parser')
            items = soup.find_all("div", class_="cassetteitem")

            if not items:
                break

            data_list = []
            seen = set()  # 重複排除用
            
            for item in items:
                try:
                    title_elem = item.find("div", class_="cassetteitem_content-title")
                    name = title_elem.text.strip() if title_elem else "不明"
                    
                    age_elem = item.find("li", class_="cassetteitem_detail-col3")
                    if age_elem:
                        age_text = age_elem.find_all("div")[0].text.strip()
                        if "新築" in age_text:
                            age = 0
                        else:
                            age_match = re.search(r'\d+', age_text)
                            age = int(age_match.group()) if age_match else 99
                    else:
                        age = 99

                    tbody = item.find("table", class_="cassetteitem_other")
                    if tbody:
                        for tr in tbody.find("tbody").find_all("tr"):
                            try:
                                tds = tr.find_all("td")
                                
                                # 価格を取得
                                price_elem = tds[3].find("li")
                                if price_elem:
                                    price_text = price_elem.text.strip()
                                    price = int(float(price_text.replace("万円", "")) * 10000)
                                else:
                                    continue

                                # 間取りを取得
                                floor_plan = tds[2].text.strip() if len(tds) > 2 else "不明"

                                # 重複チェック（名前、価格、築年数、間取りで判定）
                                key = (name, price, age, floor_plan)
                                if key not in seen:
                                    data_list.append(key)
                                    seen.add(key)
                            except Exception:
                                continue

                except Exception:
                    continue

            if data_list:
                cur.executemany("INSERT INTO properties (name, price, age, floor_plan) VALUES (?, ?, ?, ?)", data_list)
                conn.commit()
                print(f" -> {len(data_list)} 件保存")
            
            time.sleep(3)
            
        except Exception as e:
            print(f"Error: {e}")
            break

get_data()
conn.close()
print("スクレイピング完了")

1回目 
 -> 30 件保存
2回目 
 -> 30 件保存
3回目 
 -> 30 件保存
4回目 
 -> 30 件保存
5回目 
 -> 30 件保存
6回目 
 -> 30 件保存
7回目 
 -> 30 件保存
8回目 
 -> 30 件保存
9回目 
 -> 30 件保存
10回目 
 -> 30 件保存
11回目 
 -> 30 件保存
スクレイピング完了
